# N168 · 约束建模与算法选择训练

**目标：** 在不看标签时根据结构提出可检验候选。

**先修：** N016, N029, N056, N103。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 目标类型；数据性质；约束；状态图；单调性；候选排除；不是关键词套模板。

## 从问题到算法

同一个“最短连续区间和至少为 target”的问题，删除“非负”约束就改变了解法。非负数组扩张右端不会降低和，缩短左端不会增大和，才允许普通滑动窗口单向排除答案。

含负数时改用前缀和：寻找最小的 j−i，使 `P[j]−P[i]≥target`。若较新的 i 有不大的前缀值，它比更老、更大的前缀同时更易达标且更短，旧候选因此被支配。单调双端队列保存未被支配的候选。

两个公开函数都要求正 target，无解统一返回 −1。比较表明确标记不适用，不自动调用更强求解器替一个前提错误的方法“兜底”。

## 最小实现

**本章接口：** `solve_min_length_positive(nums, target)`、`solve_min_length_signed(nums, target)`、`compare_candidates(cases)`

In [1]:
from collections import deque

def solve_min_length_positive(nums, target):
    """Nonnegative values, positive target; return -1 when no nonempty window works."""
    if target <= 0 or any(x < 0 for x in nums):
        raise ValueError('Requires nonnegative elements and positive target')
    left = total = 0
    answer = len(nums)+1
    for right,value in enumerate(nums):
        total += value
        while total >= target:
            answer = min(answer,right-left+1)
            total -= nums[left]
            left += 1
    return -1 if answer > len(nums) else answer

def solve_min_length_signed(nums, target):
    if target <= 0:
        raise ValueError('Positive target required')
    prefix = [0]
    for value in nums:
        prefix.append(prefix[-1]+value)
    candidates = deque()
    answer = len(nums)+1
    for right,current in enumerate(prefix):
        while candidates and current-prefix[candidates[0]] >= target:
            answer = min(answer,right-candidates.popleft())
        while candidates and prefix[candidates[-1]] >= current:
            candidates.pop()
        candidates.append(right)
    return -1 if answer > len(nums) else answer

def compare_candidates(cases):
    rows = []
    for nums,target in cases:
        positive_result = solve_min_length_positive(nums,target) if all(x>=0 for x in nums) else 'inapplicable: negative element'
        rows.append((nums,target,positive_result,solve_min_length_signed(nums,target)))
    return rows

## 正确性、前提与复杂度

非负窗口依赖左右变化的单调性；负数反例直接否定该前提缺失时的推导。前缀队列的尾部删除基于支配关系，头部删除基于已找到该起点最短的当前终点，未来终点只会更长。每个下标最多入队、出队各一次。

**代价：** 普通窗口时间 O(n)、辅助空间 O(1)；前缀单调队列时间 O(n)、空间 O(n)。独立穷举只用于小规模核验。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['nums','target','nonnegative window','signed deque'], compare_candidates([
    ([2,3,1,2,4,3],7),([1,-1,3],3),([0,0,1],2),([],1)]))

nums,target,nonnegative window,signed deque
"[2, 3, 1, 2, 4, 3]",7,2,2
"[1, -1, 3]",3,inapplicable: negative element,1
"[0, 0, 1]",2,-1,-1
[],1,-1,-1


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
import itertools
def brute(nums,target):
    valid=[j-i for i in range(len(nums)) for j in range(i+1,len(nums)+1) if sum(nums[i:j])>=target]
    return min(valid,default=-1)
for n in range(6):
    for nums in itertools.product([-1,0,2],repeat=n):
        for target in [1,2,3,5]:
            expected=brute(nums,target)
            assert solve_min_length_signed(nums,target)==expected
            if all(x>=0 for x in nums):assert solve_min_length_positive(nums,target)==expected
assert solve_min_length_signed([1,-1,3],3)==1
try:solve_min_length_positive([1,-1,3],3)
except ValueError:pass
else:raise AssertionError('Missing nonnegative precondition')
# Deliberately incorrect outside its domain: run only to expose the counterexample.
def invalid_plain_window(nums,target):
    left=total=0;answer=len(nums)+1
    for right,value in enumerate(nums):
        total+=value
        while total>=target:
            answer=min(answer,right-left+1);total-=nums[left];left+=1
    return -1 if answer>len(nums) else answer
assert invalid_plain_window([1,-1,3],3)==3
assert brute([1,-1,3],3)==1
print('N168: 所有本章断言通过')

N168: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 去掉题目标签重新建模。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 写出每种候选算法必须成立的前提。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 209. Minimum Size Subarray Sum（canonical）
- 862. Shortest Subarray with Sum at Least K（comparison）：[官方入口](https://leetcode.com/problems/shortest-subarray-with-sum-at-least-k/)
- 300. Longest Increasing Subsequence（comparison）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。